# Introduction
This notebook will use PySpark's MLlib to build and compare supervised learning models on a gene
expression dataset for leukemia classification. First, we read in the data, split it into training and test
sets, and define an evaluation metric. We then fit three model classes (Random Forest, Multinomial Elastic Net, and Multilayer Perceptron) using pipelines and cross-validation to select the best
hyperparameters for each. Finally, we evaluate each best model on the held-out test set and determine
an overall winner.

The weighted F1 score as the primary evaluation metric because it balances precision and recall, penalizing both false positives and negatives, while accounting for class imbalance across the three diagnosis groups.

In [1]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler, PCA
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [2]:
spark = (SparkSession.builder
    .appName("GeneExpression")
    .master("local[*]")
    .config("spark.driver.memory", "8g")
    .getOrCreate())

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/09 17:47:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Dataset Overview

**Source:** [Gene Expression Dataset](https://www.kaggle.com/datasets/ziya07/gene-expression-dataset) by Ziya (Kaggle)

This dataset contains microarray gene expression profiles curated for building
computational models for early leukemia diagnosis. Each row is a patient sample
whose columns are normalised expression levels of individual genes obtained from
microarray experiments. The target column (last column) assigns each sample to one
of three diagnostic classes:

- **ALL**: Acute Lymphoblastic Leukemia
- **AML**: Acute Myeloid Leukemia
- **Healthy**: Healthy Controls

According to the dataset card the data has already been preprocessed:
* Missing values have been imputed.
* Normalisation has been applied so gene expression values are uniformly scaled.

Below is quick overview of the dimensions, schema, class balance, and presence of any
remaining null values.

In [3]:
df = spark.read.option("header", "true").option("inferSchema", "true").csv("leukemia_gene_expression.csv")

# Identify columns
label_col = df.columns[-1]
feature_cols = [c for c in df.columns if c != label_col]

In [4]:
from pyspark.sql.functions import col, sum as _sum, when
from pyspark.sql.types import DoubleType

# Dimensions
print(f"Rows:    {df.count()}")
print(f"Columns:  {len(df.columns)}")
print(f"Features: {len(feature_cols)}")
print(f"Label:    '{label_col}'\n")

# Check for non-double feature columns
non_double = [(f.name, f.dataType) for f in df.schema.fields
              if f.name in feature_cols and not isinstance(f.dataType, DoubleType)]

if non_double:
    print(f"Feature columns that are NOT DoubleType ({len(non_double)}):")
    for name, dtype in non_double:
        print(f"  {name: } {dtype}")
    print()
else:
    print("All feature columns are DoubleType.\n")

# Class distribution
print(f"Class distribution '{label_col}':")
df.groupBy(label_col).count().orderBy("count", ascending=False).show()

# Missing / null values
null_row = df.select([
    _sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).first()

bad = [(c, null_row[c]) for c in df.columns if null_row[c] > 0]

if bad:
    print(f"Columns with nulls ({len(bad)}):")
    for name, cnt in bad[:10]:
        print(f"  {name: } {cnt}")
else:
    print("No missing or NaN values found in any column.\n")

Rows:    1000
Columns:  1001
Features: 1000
Label:    'Diagnosis'

All feature columns are DoubleType.

Class distribution 'Diagnosis':
+---------+-----+
|Diagnosis|count|
+---------+-----+
|      AML|  409|
|      ALL|  380|
|  Healthy|  211|
+---------+-----+



[Stage 8:>                                                          (0 + 5) / 5]

No missing or NaN values found in any column.



# Pipeline
This classification task will use a four-stage transformation pipeline
1. `StringIndexer`: encode the diagnosis label
2. `VectorAssembler`: combine all 1000 gene features
3. `StandardScaler`: zero-mean, unit-variance
4. `PCA`: dimensionality reduction \
followed by the respective estimator for each model. \
We're also using an 80/20 split for our training and test data.

In [5]:
# Train/Test split
train, test = df.randomSplit([0.8, 0.2], seed=1)

# Pipeline: 4 transformations + 1 estimator
label_indexer = StringIndexer(inputCol=label_col, outputCol="label")
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features_raw")
scaler = StandardScaler(inputCol="features_raw", outputCol="features_scaled", withStd=True, withMean=True)
pca = PCA(inputCol="features_scaled", outputCol="features")

evaluator = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")

# Multinomial Elastic Net
This method is an extension of logistic regression to three or more classes that estimates class probabilities with a softmax function (instead of the logit/probit function), with a penalty term that blends L1 (lasso) and L2 (ridge) regularization to shrink coefficients and perform feature selection at the same time.


We'll use the four-stage transformation pipeline \
`StringIndexer → VectorAssembler → StandardScaler → PCA` \
and swap in a `LogisticRegression` with `family="multinomial"` as the estimator. \
Cross-validation searches over PCA dimensionality, regularization strength, and elastic net mixing, again evaluated on weighted F1.

In [6]:
from pyspark.ml.classification import LogisticRegression

lr = LogisticRegression(featuresCol="features", labelCol="label",
                        family="multinomial", maxIter=500)

pipeline_lr = Pipeline(stages=[label_indexer, assembler, scaler, pca, lr])

param_grid_lr = (ParamGridBuilder()
    .addGrid(pca.k, [10, 30, 50])
    .addGrid(lr.regParam, [0.001, 0.01, 0.1])
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
    .build())

cv_lr = CrossValidator(estimator=pipeline_lr, estimatorParamMaps=param_grid_lr,
                       evaluator=evaluator, numFolds=5, seed=1, parallelism=16)

cv_model_lr = cv_lr.fit(train)

# Evaluate
lr_predictions = cv_model_lr.transform(test)
print(f"Multinomial Logistic Regression — Test F1: {evaluator.evaluate(lr_predictions):.4f}")

# Best hyperparameters
best_lr = cv_model_lr.bestModel
print(f"Best PCA k:          {best_lr.stages[3].getK()}")
print(f"Best regParam:       {best_lr.stages[4].getOrDefault('regParam')}")
print(f"Best elasticNetParam:{best_lr.stages[4].getOrDefault('elasticNetParam')}")

Multinomial Logistic Regression — Test F1: 0.3036
Best PCA k:          30
Best regParam:       0.001
Best elasticNetParam:0.5


# Random Forest Classifier
This is an ensemble method that builds many decision trees, each trained on a random bootstrap sample of the data with a random subset of features at each split, then aggregates their predictions by majority vote to reduce overfitting and variance.

We'll use the four-stage transformation pipeline \
`StringIndexer → VectorAssembler → StandardScaler → PCA` \
followed by a `RandomForestClassifier` estimator. \
Cross-validation searches over PCA dimensionality, number of trees, and max tree depth, selecting the combination that maximises weighted F1.

In [7]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(featuresCol="features", labelCol="label", seed=1)

pipeline_rf = Pipeline(stages=[label_indexer, assembler, scaler, pca, rf])

# Cross-validation
param_grid = (ParamGridBuilder()
    .addGrid(pca.k, [10, 30, 50])
    .addGrid(rf.numTrees, [50, 100, 200])
    .addGrid(rf.maxDepth, [5, 10, 15])
    .build())

cv = CrossValidator(estimator=pipeline_rf, estimatorParamMaps=param_grid,
                    evaluator=evaluator, numFolds=5, seed=1, parallelism=16)

cv_model = cv.fit(train)

# Evaluate
rf_predictions = cv_model.transform(test)
print(f"Random Forest — Test F1: {evaluator.evaluate(rf_predictions):.4f}")

# Best hyperparameters
best = cv_model.bestModel
print(f"Best PCA k:    {best.stages[3].getK()}")
print(f"Best numTrees: {best.stages[4].getNumTrees}")
print(f"Best maxDepth: {best.stages[4].getOrDefault('maxDepth')}")

Random Forest — Test F1: 0.3096
Best PCA k:    10
Best numTrees: 100
Best maxDepth: 15


# Multilayer Perceptron Classifier
This method consists of a feedforward neural network composed of layers of interconnected nodes, where each node applies a weighted sum of its inputs followed by a nonlinear activation function. Then, the network learns the weights through backpropagation to capture complex, non-linear relationships in the data.

Again, we'll use the same four-stage transformation pipeline \
`StringIndexer → VectorAssembler → StandardScaler → PCA` \
followed by a `MultilayerPerceptronClassifier` estimator. \
There are a few setup considerations with the MLP classifier. \
Since the input layer size must equal the PCA output dimension \
and the output layer must equal the number of classes (3), \
the parameter grid must be manually built to keep `pca.k` and `mlp.layers` in sync. \
Cross-validation searches over PCA dimensionality and hidden-layer architectures, selecting the combination that maximises weighted F1.

In [8]:
from pyspark.ml.classification import MultilayerPerceptronClassifier

mlp = MultilayerPerceptronClassifier(featuresCol="features", labelCol="label",
                                     maxIter=200, seed=1, blockSize=128)

pipeline_mlp = Pipeline(stages=[label_indexer, assembler, scaler, pca, mlp])

# Build grid manually so pca.k and the input layer stay in sync
param_grid_mlp = [
    {pca.k: k, mlp.layers: [k] + h + [3]}
    for k in [10, 30, 50]
    for h in [[64], [128], [64, 32]]
]

cv_mlp = CrossValidator(estimator=pipeline_mlp, estimatorParamMaps=param_grid_mlp,
                        evaluator=evaluator, numFolds=5, seed=1, parallelism=16)

cv_model_mlp = cv_mlp.fit(train)

# Evaluate
mlp_predictions = cv_model_mlp.transform(test)
print(f"Multilayer Perceptron — Test F1: {evaluator.evaluate(mlp_predictions):.4f}")

# Best hyperparameters
best_mlp = cv_model_mlp.bestModel
print(f"Best PCA k:  {best_mlp.stages[3].getK()}")
print(f"Best layers: {best_mlp.stages[4].getOrDefault('layers')}")

Multilayer Perceptron — Test F1: 0.3551
Best PCA k:  30
Best layers: [30, 64, 32, 3]


# Model Evaluation
We'll evaluate each cross-validated best model on the held-out test set using weighted F1 (primary metric), accuracy, weighted precision, and weighted recall, then select the overall best model.

In [9]:
import pandas as pd

models = {
    "Multinomial Elastic Net": cv_model_lr,
    "Random Forest":           cv_model,
    "Multilayer Perceptron":   cv_model_mlp,
}

metrics = ["f1", "accuracy", "weightedPrecision", "weightedRecall"]
rows = []

for name, model in models.items():
    preds = model.transform(test)
    row = {"Model": name}
    for m in metrics:
        row[m] = round(evaluator.setMetricName(m).evaluate(preds), 4)
    rows.append(row)

results = pd.DataFrame(rows).set_index("Model")
display(results)

# Reset evaluator to primary metric
evaluator.setMetricName("f1")

best_name = results["f1"].idxmax()
print(f"\nBest overall model: {best_name} (Test F1 = {results.loc[best_name, 'f1']:.4f})")

,f1,accuracy,weightedPrecision,weightedRecall
Model,,,,
Multinomial Elastic Net,0.3036,0.3619,0.2808,0.3619
Random Forest,0.3096,0.3905,0.2969,0.3905
Multilayer Perceptron,0.3551,0.3762,0.3573,0.3762



Best overall model: Multilayer Perceptron (Test F1 = 0.3551)


All three models performed poorly, with F1 scores ranging from 0.30 to 0.36. The features provide limited power in discriminating the classes and only marginally exceed random guessing for a three-class problem (1/3). The Multilayer Perceptron achieved the highest F1 of 0.3551, likely due to its ability to capture non-linear relationships that the linear Elastic Net and tree-based Random Forest could not effectively learn.